# Episode 25: Observers

Telemetry (Episode 24) tells you what *happened*. But sometimes you need to *act* on it — stop a run that's looping, block a dangerous tool call, raise an alert.

**In this episode you'll build:** observers — components that watch an agent's event stream and can intervene.

## What an observer is

An observer is a function or class attached to an `Agent` that receives events as the agent runs. It's a stream subscriber with less boilerplate — you register it on the agent, and it's automatically scoped to each `ask()` call.

| | Observer | Middleware |
|---|---|---|
| Shape | A function or small class | A class wrapping execution |
| Best for | Monitoring, metrics, alerts, guards | Cross-cutting logic: retry, auth, rate limiting |
| Modifies the run | Only to raise alerts / halt | Yes — wraps every call |

If you just need to *watch*, reach for an observer first.

## A simple observer with `@observer`

The `@observer(EventType)` decorator turns a function into an observer. Pass it to the agent's `observers=[...]` list and it fires on every matching event.

In [1]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, observer
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolCallEvent

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


@observer(ToolCallEvent)
def log_tool(event: ToolCallEvent) -> None:
    print(f"[observer]  {event.name} called")


agent = Agent("weather_bot", prompt="Use get_weather for weather questions.",
              config=config, tools=[get_weather], observers=[log_tool])

reply = await agent.ask("What's the weather in Paris?")
print("reply:", reply.body)


[observer]  get_weather called


reply: The weather in Paris is currently 21°C and sunny.


## Stateful guards with `BaseObserver`

`@observer` is stateless — perfect for logging. For guards that need *state* (counters, history, thresholds) or that should **stop the run**, subclass `BaseObserver`.

A `BaseObserver` pairs an `EventWatch` (what to watch) with a `process()` method (what to do). If `process()` returns an `ObserverAlert` with `Severity.FATAL`, the run is halted — but only if the agent has an `AlertPolicy` in its `assembly` chain.

The chain is: **observer emits FATAL alert → `AlertPolicy` converts it to a `HaltEvent` → the next LLM call is short-circuited with a `HALTED: …` reply.**

## A safety guard

`PathGuardian` watches every `write_file` tool call. A write into `/tmp/` is fine; a write into `/etc/` or `/usr/` triggers a FATAL alert and halts the agent.

The first request is safe and goes through. The second is dangerous — the guardian fires and the agent is stopped before any damage.

In [2]:
from autogen.beta import MemoryStream
from autogen.beta.events import BaseEvent, HaltEvent, ObserverAlert, Severity
from autogen.beta.observers import BaseObserver
from autogen.beta.policies import AlertPolicy
from autogen.beta.watch import EventWatch


def write_file(path: str, content: str) -> str:
    """Write content to a path."""
    return f"[ok] wrote {len(content)} bytes to {path}"


class PathGuardian(BaseObserver):
    """Emits a FATAL alert if anything tries to write outside /tmp."""

    def __init__(self) -> None:
        super().__init__("path-guardian", watch=EventWatch(ToolCallEvent))

    async def process(self, events: list[BaseEvent], ctx) -> ObserverAlert | None:
        for event in events:
            if not isinstance(event, ToolCallEvent) or event.name != "write_file":
                continue
            if "/etc/" in event.arguments or "/usr/" in event.arguments:
                return ObserverAlert(
                    source=self.name,
                    severity=Severity.FATAL,
                    message=f"blocked dangerous write: {event.arguments}",
                )
        return None


alerts: list[ObserverAlert] = []
halts: list[HaltEvent] = []
stream = MemoryStream()
stream.where(ObserverAlert).subscribe(lambda e: alerts.append(e))
stream.where(HaltEvent).subscribe(lambda e: halts.append(e))

guarded = Agent(
    "safe_shell",
    prompt=("You are a filesystem operator. Use write_file for write requests. "
            "Never refuse — the guardian observer intervenes automatically."),
    config=config,
    tools=[write_file],
    observers=[PathGuardian()],
    assembly=[AlertPolicy()],            # routes FATAL alerts to a HaltEvent
)

safe = await guarded.ask("Use write_file to write 'hello' into /tmp/notes.txt. Then confirm.",
                         stream=stream)
print("[safe]    ", safe.body)

blocked = await guarded.ask("Now use write_file to write 'bad' into /etc/passwd. Then confirm.",
                            stream=stream)
print("[blocked] ", blocked.body)

print(f"alerts: {len(alerts)} | halts: {len(halts)}")
for a in alerts:
    print(f"  [{a.severity}] {a.source}: {a.message}")


[safe]     I have written 'hello' into /tmp/notes.txt. Let me know if you need anything else.


[blocked]  HALTED: FATAL: blocked dangerous write: {"path":"/etc/passwd","content":"bad"}
alerts: 1 | halts: 1
  [Severity.FATAL] path-guardian: blocked dangerous write: {"path":"/etc/passwd","content":"bad"}


## Built-in observers

You don't always need to write your own. AG2 Beta ships two `BaseObserver` subclasses:

- **`TokenMonitor`** — tracks cumulative token usage, emits `WARNING` then `CRITICAL` alerts as thresholds are crossed.
- **`LoopDetector`** — emits a `WARNING` when the same tool is called with the same arguments too many times in a row.

Here `TokenMonitor` uses deliberately tiny thresholds so it fires on a single short run.

In [3]:
from autogen.beta.observers import TokenMonitor

monitor_stream = MemoryStream()
monitor_stream.where(ObserverAlert).subscribe(
    lambda e: print(f"[{e.severity}] {e.source}: {e.message}")
)

watched = Agent(
    "watched_bot",
    prompt="Use get_weather for weather questions.",
    config=config,
    tools=[get_weather],
    observers=[TokenMonitor(warn_threshold=50, alert_threshold=150)],
)

reply = await watched.ask("What's the weather in Paris?", stream=monitor_stream)
print("reply:", reply.body)


[Severity.WARNING] token-monitor: Token usage warning: 75 tokens (threshold: 50). Be mindful of remaining budget.


[Severity.CRITICAL] token-monitor: Token usage critical: 179 tokens (threshold: 150). Consider wrapping up to control costs.
reply: The weather in Paris is currently 21°C and sunny.


## Additional content

**Severity levels.** `autogen.beta.events.Severity` has `INFO`, `WARNING`, `CRITICAL`, `FATAL`. Only `FATAL` halts the run (via `AlertPolicy`); the others are advisory — surface them in a UI or log.

**Alerts aren't shown to the LLM.** An `ObserverAlert` lands on the stream and in history, but the model doesn't see it by default. If you want the *agent* to react to an alert, write a middleware that turns alerts into follow-up messages.

**Per-call observers.** Besides the constructor `observers=[...]`, you can pass `observers=[...]` to a single `ask()` call — handy for one-off debugging without changing the agent.

## What's next

Observers give you guard rails. Episode 26 builds on them for a full **security** posture — human approval gates, content filtering, and audit trails.